# 12-тарау. Трансформер және GPT архитектурасы

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 12.1. Трансформердің құрылысы

In [ ]:
import math
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F

ln = nn.LayerNorm(4)
x = torch.tensor([[1., 2., 3., 10.]])
y = ln(x)
print(y)
print(f"орташа={y.mean().item():.3f}, "
      f"ауытқу={y.std(unbiased=False).item():.3f}")

## 12.2. GPT-ті жазамыз

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 100     # сөздік өлшемі
    block_size: int = 64      # контекст ұзындығы
    n_layer: int = 4          # блоктар саны
    n_head: int = 4           # назар бастарының саны
    n_embd: int = 128         # вектор өлшемі
    dropout: float = 0.1

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head)
                    .transpose(1, 2)
                   for t in (q, k, v)]
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd), nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # назар аудару + қалдық байланыс
        x = x + self.mlp(self.ln2(x))       # MLP + қалдық байланыс
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(
            [Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight   # салмақтарды ортақтастыру
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

## 12.3. Модельді тексереміз

In [ ]:
torch.manual_seed(0)
model = GPT(GPTConfig(vocab_size=100))

def count(module):
    return sum(p.numel() for p in module.parameters())

print(f"Барлық параметр: {count(model):,}".replace(",", " "))
print("эмбеддингтер:", count(model.tok_emb) + count(model.pos_emb))
print("бір блок:", count(model.blocks[0]),
      "(назар:", count(model.blocks[0].attn),
      "MLP:", str(count(model.blocks[0].mlp)) + ")")

idx = torch.randint(0, 100, (2, 16))       # 2 мысал, 16 токен
targets = torch.randint(0, 100, (2, 16))   # кездейсоқ «дұрыс жауаптар»
logits, loss = model(idx, targets)
print(logits.shape)
print(f"Бастапқы шығын: {loss.item():.3f}, ln(100) = {math.log(100):.3f}")

In [ ]:
gpt2 = GPT(GPTConfig(vocab_size=50257, block_size=1024,
                     n_layer=12, n_head=12, n_embd=768))
print(f"GPT-2 (кіші): {count(gpt2) / 1e6:.1f} млн параметр")

## 12.4. Мәтін генерациялау

In [ ]:
@torch.no_grad()
def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -self.cfg.block_size:]      # соңғы контекст
        logits, _ = self(idx_cond)
        logits = logits[:, -1, :] / temperature       # соңғы позиция
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx

GPT.generate = generate      # әдісті класқа қосамыз
torch.manual_seed(0)
out = model.generate(torch.zeros(1, 1, dtype=torch.long), max_new_tokens=15)
print(out.tolist())